# Prizma — ONE-SESSION GPU CAMPAIGN notebook (A100)

**Purpose:** ONE Colab A100 session (spanning as many 24h Colab sessions as the stages need)
executes the ENTIRE GPU-tier program **in the locked order** (commission synthesis §6, extended):
Tier-0 repairs → PR-01 powered → PR-02 powered → **PR-08 (Prizma-LM flagship bar) powered** — then archives and zips everything for download.

**PRE-REGISTRATION DISCIPLINE (read before Run all):**
- The claim runners execute **frozen, pre-registered protocols** (`docs/preregistry/`,
  `docs/crosstalk_capacity_law.md`). **Do not edit knobs, grids, seeds, steps, or paths.**
  The runners compute their own verdicts, including the pre-committed negative branches
  (RETIRED / DEMOTED / INCOMPLETE / FALSIFIED). A runner REFUSING to run (CUDA guard, ledger
  guard, instrument drift) is the protocol working — do not bypass it.
- Raw-first retention (`docs/RETENTION.md`): every runner streams crash-safe JSON ledgers and
  archives raw records BEFORE any verdict. Artifacts are only ever COPIED by this notebook,
  never moved or deleted.
- **Smoke vs powered:** every claim runner has a `--smoke` (CPU plumbing, meaningless numbers)
  and a `--powered` (the registered campaign, refuses without CUDA) mode. This notebook runs
  `--powered` only. Never cite smoke numbers.

**HONEST WALL-TIME ESTIMATES** (from the frozen docs / RALPH ledger — estimates, not promises;
Prizma runs have historically come in 2-15× faster than estimated, e.g. PR-07′ ~9 min vs 2-3h):

| # | Stage | Command | Est. wall time | Colab 24h limit |
|---|-------|---------|----------------|-----------------|
| 0 | Env setup + pytest sanity | cells below | ~10 min | fine |
| 1 | Tier-0: clean n=10 recall gate | `python -m seq.recall_gate --full` | **~28 h** | EXCEEDS → resume across sessions (below) |
| 2 | Tier-0: B4 closure (char-LM, both corpora, n=5) | `python gpu_charlm2.py --corpus {text8,shakespeare} --seeds 0 1 2 3 4` | ~8 h total | borderline → resume works |
| 3 | Tier-0: GLA/Mamba-2 landscape | `python seq/landscape.py --full` | **~60 h** | EXCEEDS → resume across sessions |
| 4 | PR-01 powered | `python seq/surprise_claim.py --powered` | ~10–15 A100-h | borderline → resume works |
| 5 | PR-02 powered | `python seq/dfrontier_claim.py --powered` | 27 fit + 36 adjudication runs ≈ **12–18 A100-h**; fit tier alone 3–5 h | borderline → resume works; K1/K2 may legitimately stop it at the fit tier |
| 6 | PR-08 (Prizma-LM flagship) powered | `python seq/prizma_lm_claim.py --powered --trunk-lr-c 7.5e-4 --domain-exclusion --ledger-dir prizma_lm_PR-2026-09-03-13_gpu` (the REPAIRED flagship — PR-13 CLAIMED 2026-09-09 on CPU; this A100 stage is CONFIRMATORY, own ledger, not gating) | ~45–90 min (CPU-measured pace: PR-07′ ran 9 min vs 2–3 h estimate) | fine |
| 7 | Archive + zip to Drive | final cells | minutes | fine |

**RESUME DISCIPLINE (verified in the runners' code):** every stage streams its ledger to
`$PRIZMA_RESULTS` (Drive) and caches every training cell under a `(cellkey, config-fingerprint)`
resume key (`seq/gpu_harness.run_cell` / `sweep_then_seeds`; `seq/recall_gate.py` uses the same
machinery). If a session dies at hour N: open a fresh session, re-run cells 1–4 (setup), then
re-run the SAME stage cell with the SAME command — completed cells are reused and training
continues where it stopped. **The resume command is always the identical command.**

**Expected pytest sanity:** `569 collected -> 559 passed, 10 skipped` on CPU (updated 2026-09-11;
the count only grows via tested additions — record the ACTUAL output).
ANY failure/error = STOP; do not run the claim stages on a broken tree.


In [ ]:
# Stage 0a — GPU check (A100). Everything downstream refuses without CUDA.
import subprocess
import torch

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "nvidia-smi gave no output")
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > A100 GPU"


In [ ]:
# Stage 0b — mount Drive, point $PRIZMA_RESULTS at it (crash-safe ledgers SURVIVE disconnects),
# and define the per-stage archive helper (COPY-only: docs/RETENTION.md).
import glob
import os
import shutil
import time

from google.colab import drive

drive.mount("/content/drive")
os.environ["PRIZMA_RESULTS"] = "/content/drive/MyDrive/prizma_results"
os.makedirs(os.environ["PRIZMA_RESULTS"], exist_ok=True)
print("PRIZMA_RESULTS =", os.environ["PRIZMA_RESULTS"])


def archive_stage(stage, patterns):
    """COPY this stage's artifacts into <results>/campaign_archives/<stage>-<utc>/ (never move or
    delete raw artifacts — docs/RETENTION.md). `patterns` are repo-root-relative-or-absolute globs
    under the Drive results root."""
    root = os.environ["PRIZMA_RESULTS"]
    stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
    arc = os.path.join(root, "campaign_archives", f"{stage}-{stamp}")
    os.makedirs(arc, exist_ok=True)
    n = 0
    for pat in patterns:
        for src in glob.glob(os.path.join(root, pat)):
            dst = os.path.join(arc, os.path.basename(src.rstrip("/")))
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
            n += 1
    print(f"archived {n} paths -> {arc}")
    return arc


In [ ]:
# Stage 0c — get the repo. EITHER clone (recommended, needs network) OR upload a zip of the repo
# to /content/ and unzip it into /content/Prizma (then skip the clone line).
!test -d /content/Prizma/.git || git clone https://github.com/nazmiefearmutcu/Prizma /content/Prizma
%cd /content/Prizma
!git log --oneline -3


In [ ]:
# Stage 0d — deps + pytest sanity gate.
# torch arrives PREINSTALLED on Colab as a CUDA wheel: do NOT downgrade it to the CPU wheel here.
# (On a CPU-only box the recipe is instead:  pip install torch --index-url https://download.pytorch.org/whl/cpu )
# EXPECT the sanity line: "569 collected -> 559 passed, 10 skipped" (updated 2026-09-11; the count only
# grows via tested additions — record the ACTUAL output). ANY failed/error = STOP here.
!pip install -q -r requirements.txt
!python -m pytest -q 2>&1 | tail -3


## Stage 1 — Tier-0 repair: clean n=10 recall gate  (~28 h; resume across sessions)

Frozen runner: `seq/recall_gate.py --full` (BAR-0 smoke/campaign ledger separation, TOST-parity
gate, n=10). **~28 h exceeds Colab's 24 h limit by design** — verified: the runner caches every
training cell under `(cellkey, config-fingerprint)` in the Drive ledger, so the resume procedure
is: fresh session → re-run setup cells → re-run THIS EXACT COMMAND. It continues where it stopped.


In [ ]:
!python -m seq.recall_gate --full


In [ ]:
# Stage 1 archive (copy-only)
archive_stage("tier0_recall_gate", ["recall_gate.json", "runs/recall_gate-*.json"])


## Stage 2 — Tier-0 repair: B4 closure — credible char-LM, BOTH corpora, n=5  (~8 h total)

Frozen runner: `gpu_charlm2.py` (regularized Prizma-quad2 vs Transformer, param-matched, test BPC;
crash-safe ledger `$PRIZMA_RESULTS/gpu_charlm2.json`, resume = re-run the same command).
CLI wiring documented (2026-09-07): `--seeds` **defaults to [0, 1]** — the n≥5 closure MUST pass
`--seeds 0 1 2 3 4` explicitly; the tiny-shakespeare leg needs `--corpus shakespeare` (the default
corpus is text8 — without the flag you would silently re-run text8). Corpora auto-download from
their canonical URLs and cache under `$PRIZMA_RESULTS` (karpathy char-rnn for tiny-shakespeare;
text8 HF-raw → zip fallback).


In [ ]:
!python gpu_charlm2.py --corpus text8 --seeds 0 1 2 3 4


In [ ]:
!python gpu_charlm2.py --corpus shakespeare --seeds 0 1 2 3 4


In [ ]:
# Stage 2 archive (copy-only)
archive_stage("tier0_b4_charlm", ["gpu_charlm2.json"])


## Stage 3 — Tier-0 repair: GLA/Mamba-2 powered landscape  (~60 h; multi-session)

Frozen runner: `seq/landscape.py --full` — the recall-leg 4-arm head-to-head
(TF vs Prizma-v2 vs GLA vs Mamba-2) with the identical-model negative control. (The char-LM BPC
leg `--charlm` is a separate leg, not part of this Tier-0 item.) Multi-session by design:
resume = re-run THIS EXACT COMMAND (sweep_then_seeds caches by `(cellkey, config-fingerprint)`).


In [ ]:
!python seq/landscape.py --full


In [ ]:
# Stage 3 archive (copy-only)
archive_stage("tier0_landscape", ["gpu_landscape.json"])


## Stage 4 — PR-2026-09-03-01 POWERED: surprise-gating ablation (retire-if-negative)  (~10–15 A100-h)

`python seq/surprise_claim.py --powered` executes the frozen pre-registration
(`docs/preregistry/2026-09-03-surprise-gating-powered-ablation.md`) VERBATIM: 4 arms × 3 tasks,
n=5 claim seeds, per-(arm×task) LR fairness sweeps, the A4 NO-TUNING gain selection (exploratory
seed 900 → its own LANE-EXPLORATORY ledger), the identical-arm integrity canary (a FAIL invalidates
the campaign → INCONCLUSIVE, no claim), and the frozen Welch+Holm verdict including the
pre-committed RETIRED branch. Run AFTER the Tier-0 stages, BEFORE PR-02 (locked order).


In [ ]:
!python seq/surprise_claim.py --powered


In [ ]:
# Stage 4 archive (copy-only; the gain-selection file is LANE-EXPLORATORY, archived for honesty)
archive_stage("pr01_surprise_ablation",
              ["surprise_ablation_PR-2026-09-03-01", "exploratory/surprise_gain_selection.json"])


## Stage 5 — PR-2026-09-03-02 POWERED: the registered D-frontier grid (capacity-law adjudicator)

`python seq/dfrontier_claim.py --powered` executes `docs/crosstalk_capacity_law.md` §4–§5 VERBATIM:
- **Fit** ε ONCE at D ∈ {16, 32, 64} (3 arms × 3 rungs × 3 seeds = 27 runs, ~3–5 A100-h). The fit
  tier alone is the cheap decisive filter: **K1** (ε̂ ∉ [0.25, 4.0] → DEMOTED) or **K2** (primary
  solves-all/fails-all → INCOMPLETE) legitimately STOP the run right there — that is a protocol
  outcome, not a failure.
- **FREEZE**: predictions (N* per arm, per-rung solve/not-solve at D ∈ {96, 128, 192, 256}) are
  written into the ledger BEFORE any adjudication cell; a resumed run reuses them and refuses to
  re-fit.
- **Adjudicate** (36 runs, ~9–13 A100-h): frozen PASS bar = observed per-rung solve matches the
  frozen prediction on **≥ 3 of 4 rungs** (primary `quad2`); **K4** below the bar falsifies the
  law as the predictive account; **K3** non-monotone frontier → one fresh-seed re-run, persists →
  INCOMPLETE. `none` @ d_φ=32 is the free sanity arm (predicted to fail ALL ≥96 rungs).


In [ ]:
!python seq/dfrontier_claim.py --powered


In [ ]:
# Stage 5 archive (copy-only)
archive_stage("pr02_dfrontier", ["dfrontier_PR-2026-09-03-02"])


## Stage 6 — PR-13 CONFIRMATORY: the REPAIRED Prizma-LM flagship (many-block fused column)  (~45–90 A100-min)

`python seq/prizma_lm_claim.py --powered --trunk-lr-c 7.5e-4 --domain-exclusion --ledger-dir prizma_lm_PR-2026-09-03-13_gpu`
executes `docs/preregistry/2026-09-08-prizma-lm-flagship.md` with the two REGISTERED REPAIR LEVERS
composed — PR-10 C-block trunk-lr scaling (×0.25) + PR-12 domain-exclusive experts: 5 arms
(PRIM-LM / FROZEN-TRUNK / SHARED-HEAD / FROZEN-CHECKPOINT / FORCED-RECRUIT) × 5 seeds (0–4) over the
3-block stream text8[0:1M) → tiny-shakespeare → text8[1.1M:2.1M) with the PINNED eval slices
(A-eval [1.0M,1.1M), B-eval = shakespeare last 10%, C-retention [0.9M,1.0M)). Bars B1–B4 EXACT:
Holm over B1–B2 (upper-tail t_isf), B3 = routing ledger + forced-recruit cost, B4 = trunk-drift
accounting (reported, not gated); INCONCLUSIVE straddling-CI rule; §5 failure branches echoed in the
verdict. This A100 stage is CONFIRMATORY, not gating — PR-13 already CLAIMED all three bars on CPU
(2026-09-09) and PR-18 replicated the verdict at n=10 fresh seeds (5–14); the run uses its own ledger
(`prizma_lm_PR-2026-09-03-13_gpu`). Runs AFTER the PR-02 stage, BEFORE the final archive cell.

Disclosed (ledger meta carries both notes): (1) maintainer addendum 2026-09-08 (Pre-GPU review H-1) —
the doc's literal C = text8[1.0M,2.0M) would train on the A-eval slice and corrupt B1; the runner
pins C = [1.1M,2.1M) (probe-2 precedent). (2) §2 prose bug — C-retention [0.9M,1.0M)
IS A-train's tail; slice implemented as pinned, B2 stays a fair comparison.

In [ ]:
!python seq/prizma_lm_claim.py --powered --trunk-lr-c 7.5e-4 --domain-exclusion --ledger-dir prizma_lm_PR-2026-09-03-13_gpu

In [ ]:
# Stage 6 archive (copy-only)
archive_stage("pr13_prizma_lm", ["prizma_lm_PR-2026-09-03-13_gpu"])

## Final — zip the whole results root for Drive download

Everything already lives on Drive (`$PRIZMA_RESULTS`); this cell additionally produces ONE zip
(ledgers + raw archives + per-stage archives) and offers a browser download fallback.


In [ ]:
# Final archive — ONE zip of the entire campaign results root.
import os
import shutil
import time

root = os.environ["PRIZMA_RESULTS"]
name = time.strftime("prizma_gpu_campaign_%Y%m%dT%H%M%SZ", time.gmtime())
tmp_zip = shutil.make_archive(f"/content/{name}", "zip", root)   # built OUTSIDE the tree it zips
arc_dir = os.path.join(root, "campaign_archives")
os.makedirs(arc_dir, exist_ok=True)
dst = os.path.join(arc_dir, os.path.basename(tmp_zip))
shutil.copy2(tmp_zip, dst)
print("zip:", dst, f"({os.path.getsize(dst) / 1e6:.1f} MB) — also kept at {tmp_zip}")
try:
    from google.colab import files
    files.download(tmp_zip)   # browser fallback; harmless if dismissed
except Exception as e:        # Drive copy above is the primary delivery path
    print("browser download skipped:", e)
